In [0]:
use schema retail_dev_schema;

In [0]:
-- %python
-- | # | Problem statement | Expected output | Practise |
-- |---|---|---|---|
-- | **1** | Create a CTE counting users in each batch. From that CTE, return batches containing more than 100 records. | `batch_id`, `user_count` | CTE + aggregation |
-- | **2** | Create a CTE extracting the numeric batch suffix for each product. Use another CTE to number versions per product, newest first. Return one latest record per product ID. | Product columns | Multiple CTEs + `ROW_NUMBER` |
-- | **3** | Return one latest cart per cart ID across all batches. Keep its nested products array intact. Assume IDs are unique within each batch. | `id`, `user_id`, `total`, `batch_id` | Partition by business key |
-- | **4** | Return products at the **top three distinct prices in each category**. Include all products tied at those prices; exclude null prices. | `category`, `id`, `title`, `price`, `price_rank` | `DENSE_RANK` |
-- | **5** | Return **exactly two highest-priced products per category**, or all products if fewer than two exist. Resolve price ties using product ID ascending; exclude null prices. | `category`, `id`, `title`, `price`, `rn` | `ROW_NUMBER` + tie-breaker |
-- | **6** | Show each product’s price alongside its category’s average price. Then return products priced above that average. | `id`, `title`, `category`, `price`, `category_avg_price` | `AVG() OVER` + CTE |
-- | **7** | For each user, order carts by cart ID ascending. Show the previous cart’s total and the difference between current and previous totals. The first cart should have null previous total and difference. | `user_id`, `id`, `total`, `previous_total`, `difference` | `LAG` |
-- | **8** | Calculate cumulative cart value for each user, ordered by cart ID ascending. Explicitly use a row-based window frame. | `user_id`, `id`, `total`, `running_total` | `SUM() OVER` + window frame |

In [0]:
-- batch_id | id | occurrence_count
select batch_id, id, count(*) from bronze_users
group by batch_id, id
having count(*) > 1
order by batch_id, id;

In [0]:
WITH ranked_users AS (
    SELECT *,
           ROW_NUMBER() OVER (
               PARTITION BY id
               ORDER BY CAST(
                   REGEXP_EXTRACT(batch_id, 'batch_([0-9]+)', 1)
                   AS INT
               ) DESC
           ) AS rn
    FROM bronze_users
)
SELECT *
FROM ranked_users
WHERE rn = 1;

In [0]:
with agg_count_rec as (
    select batch_id, count(*) from bronze_users
    group by batch_id
    having count(*) > 100
)

select * from agg_count_rec;

In [0]:
select * from bronze_products;

In [0]:
WITH product_batches AS (
    SELECT *,
           CAST(
               REGEXP_EXTRACT(batch_id, 'batch_([0-9]+)', 1)
               AS INT
           ) AS batch_number
    FROM bronze_products
),
ranked_products AS (
    SELECT *,
           ROW_NUMBER() OVER (
               PARTITION BY id
               ORDER BY batch_number DESC
           ) AS rn
    FROM product_batches
)
SELECT *
FROM ranked_products
WHERE rn = 1;

In [0]:
select * from bronze_carts;